### Create Training data for binary classification: description of business sector - True False

In [2]:
import pandas as pd
import glob
import os
import ast
import json

import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
import random
from typing import List
from sklearn.model_selection import train_test_split

# sys.path.append("../../")
# from sentence_splitter import split_text_into_sentences
from __future__ import annotations

import os
from pathlib import Path

import fitz  # PyMuPDF

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption

#from BERT_classifier.Classify_report_with_BERT import classification_report_BERT
#from transformers import AutoModelForSequenceClassification, AutoTokenizer

#### Load Positives

In [12]:
positives_json_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/JSONs"
positives_json_path = "projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_3/JSONs"
positives_pdfs_path = "projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/PDFs"
description_page_path = "projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/company_descriptions"

In [6]:
#df_overview = pd.read_csv("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv", sep=";")
df_overview_3 = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_3/reports_subset_from_full_data_3_for_backtracking.csv", sep=";")
df_overview_3 = df_overview_3[df_overview_3["description_page"].notna()]
df_overview_3

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,...,Sec is Primary Issue,Sec Type,SEDOL,NACE_Letter,Report,NACE_lvl_3,description_page,Unnamed: 25,Description,LLM_summary
0,0,0,103,FR0013183985,Gensight Biologics SA,1,2012.0,FRA,F4374K117,20160713.0,...,1,SHARE,BD07M05,C,Gensight Biologics SA1.pdf,21.2,1,NaN,<!-- image -->\n\n## GenSight Biologics Report...,GenSight Biologics focuses on developing and c...
1,1,1,462,GB00BYN5BY03,Hybrid Software Group PLC,1,2013.0,BEL,G3971B107,20010417.0,...,1,SHARE,BYN5BY0,J,Hybrid Software Group PLC1.pdf,58.2,2,NaN,## CONTENTS\n\n| Hybrid Software Group ...,Hybrid Software Group PLC is a public company ...
2,2,2,651,BMG8766E1093,Textainer Group Holdings Limited,1,1979.0,USA,G8766E109,20191211.0,...,0,SHARE,BKDZ8P2,N,Textainer Group Holdings Limited1.pdf,77.3,2,NaN,<!-- image -->\n\n## About Us\n\nTextainer Gro...,Textainer Group Holdings Limited is a leading ...
3,3,3,539,AEA005901011,Amanat Holding PJSC,1,2014.0,ARE,M08598100,20141201.0,...,1,SHARE,BSZM277,K,Amanat Holding PJSC1.pdf,64.9,3,NaN,## About Amanat\n\nAmanat Holdings ( Amanat or...,Amanat Holdings is the largest investment comp...
4,4,4,530,US13738M1036,Cancer Capital Corp.,1,1997.0,USA,13738M103,20111017.0,...,1,SHARE,B6TVP88,K,Cancer Capital Corp.2.pdf,64.9,3,NaN,2\n\n2\n\n<!-- image -->\n\n## Cancer Care Wes...,Cancer Care West provides free community-based...
5,5,5,21,AU000000CSS3,Clean Seas Seafood Limited,1,2000.0,AUS,Q2508T119,20051212.0,...,1,SHARE,B0PFW92,A,Clean Seas Seafood Limited2.pdf,3.2,3,NaN,For personal use only\n\n<!-- image -->\n\n## ...,Clean Seas specializes in the full cycle breed...
6,6,6,532,US27579R1041,"East West Bancorp, Inc.",1,1998.0,USA,27579R104,19990208.0,...,1,SHARE,2487407,K,"East West Bancorp, Inc.1.pdf",64.1,3,NaN,<!-- image -->\n\nE ast West Banking Corporat...,EastWest Banking Corporation is a major univer...
7,7,7,115,MYQ0128OO007,Frontken Corp. Bhd.,1,1996.0,MYS,Y26510100,20060711.0,...,1,SHARE,B18TLC4,C,Frontken Corp. Bhd.1.pdf,26.1,13,NaN,## CHAPTER 1.0 INTRODUCTION ## 1.1 ABOUT FRONT...,Frontken Group's business model focuses on int...
8,8,8,672,AU000000MMS5,Mcmillan Shakespeare Limited,1,1988.0,AUS,Q58998107,20040315.0,...,1,SHARE,B00G1Q0,N,Mcmillan Shakespeare Limited1.pdf,77.1,3,NaN,MMS ANNUAL REPORT 2022\n\nB\n\n## Annual Gene...,McMillan Shakespeare Group provides salary pac...
9,9,9,496,US6075251024,"Model N, Inc.",1,1999.0,USA,607525102,20130320.0,...,1,SHARE,B94Z434,J,"Model N, Inc.1.pdf",58.2,3,NaN,## Introduction\n\nActivities that impact reve...,The business model focuses on optimizing reven...


In [7]:
df_overview_2 = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_overview.csv", sep=",")
df_overview_2

,Symbol,Description_page,Unnamed: 0,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,...,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3,NACE_lvl_2
0,CA05335P1099,NaN,0,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,...,Auxly Cannabis Group Inc.,1.0,XLY-CA,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf,1.3,1
1,JP3947800003,NaN,1,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,...,"MEGMILK SNOW BRAND Co., Ltd.",1.0,MMSBF-US,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf",1.4,1
2,ID1000167901,NaN,2,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,...,PT Cilacap Samudera Fishing Industry Tbk,1.0,ASHA-ID,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf,3.1,3
3,JP3843250006,NaN,3,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,...,Hokuto Corporation,1.0,1379-JP,1,SHARE,6432715,A,Hokuto Corporation1.pdf,1.3,1
4,VN000000VTQ6,NaN,4,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,...,Viet Trung Quang Binh Joint Stock Co,1.0,VTQ-VN,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf,2.3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3109,KYG4707P1054,NaN,3278,Icon Culture Global Company Limited,1,2019.0,HKG,G4707P105,20200114.0,CHN,...,Icon Culture Global Company Limited,1.0,8500-HK,1,SHARE,BJYFY12,T,Icon Culture Global Company Limited2.pdf,98.1,98
3110,ZAE000015277,NaN,12259,Brimstone Investment Corporation Limited,1,1995.0,ZAF,S13750112,19980708.0,ZAF,...,Brimstone Investment Corporation Limited,1.0,BRT-ZA,0,SHARE,6119966,A,Brimstone Investment Corporation Limited1.pdf,3.1,3
3111,NO0011013765,NaN,31130,Gigante Salmon AS,1,2001.0,NOR,R2724U105,20210705.0,NOR,...,Gigante Salmon AS,1.0,GIGA-NO,1,SHARE,BMC4Z19,A,Gigante Salmon AS1.pdf,3.1,3
3112,FR0010776617,NaN,68675,Sapmer SA,1,1989.0,FRA,F7887Q109,20090708.0,FRA,...,Sapmer SA,1.0,ALMER-FR,1,SHARE,B3LS274,A,Sapmer SA2.pdf,3.1,3


In [ ]:
# for i, row in df_overview_2.iterrows(): 
#     if pd.isna(row["Description_page"]):
#         continue

#     # if (row["Description_page"].strip() == ""):
#     #     continue

#     path = row["Report"].replace(".pdf", ".json")
    
#     with open(os.path.join(positives_json_path, path), "r") as f: 
#         report_json = json.load(f)
    
#     description_pages = [int(row["Description_page"])]

#     description_text = ""
#     for description_page in description_pages:
#         description_text += list(filter(lambda x: x["page"] == description_page + 1, report_json["pages"]))[0]["markdown"]

#     print(row["Report"])
#     print(description_text)
#     print("-----" * 5 + "\n"*6)

#     df_positives.loc[i, "Description"] = description_text

## Load description pages with docling

In [8]:
pipeline_options = PdfPipelineOptions(
    # your current options
)
# choose GPU device
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=8, 
    device=AcceleratorDevice.CUDA 
)
pipeline_options = PdfPipelineOptions(do_table_structure=True)
#pipeline_options.table_structure_options.mode = TableFormerMode.FAST

# assume you already built pipeline_options above
doc_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

In [28]:
def convert_via_single_page_pdf(
    pdf_path: str,
    page: int,
    out_dir: str = "single_pages",
    *,
    overwrite: bool = False,
):
    """
    1) Copies exactly one page from `pdf_path` into a new PDF saved under `out_dir`
    2) Converts that new single-page PDF with Docling (converting the full PDF, which is only that page)

    Args:
        pdf_path: Path to source PDF
        page: Page number to extract (1-based)
        out_dir: Folder to store single-page PDFs
        overwrite: If False, raise if output file already exists

    Returns:
        (docling_conversion_result, single_page_pdf_path)
    """
    if page < 1:
        raise ValueError("`page` must be >= 1 (1-based indexing).")

    src_path = Path(pdf_path)
    if not src_path.exists():
        raise FileNotFoundError(f"PDF not found: {src_path}")

    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # Build output filename: <stem>__p0005.pdf
    out_path = Path(out_dir) / f"{src_path.stem}__p{int(page)}.pdf"
    if out_path.exists() and not overwrite:
        print(f"Output already exists: {out_path} (set overwrite=True to replace)")
        return None, None
    

    # --- 1) Copy one page into new PDF ---
    try:
        with fitz.open(src_path) as src:
            if page > src.page_count:
                raise ValueError(f"Requested page {page} exceeds page_count={src.page_count} for {src_path}")

            dst = fitz.open()  # new empty PDF
            # insert_pdf uses 0-based indices for pages
            dst.insert_pdf(src, from_page=page - 1, to_page=page - 1)
            dst.save(out_path)
            dst.close()
    except:
        return None, None


    # --- 2) Convert the new single-page PDF (no page_range needed) ---
    pipeline_options = PdfPipelineOptions()  # default: parse full PDF (which is only one page)
    doc_converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )

    conversion_result = doc_converter.convert(str(out_path))

    md = conversion_result.document.export_to_markdown()
    return md, str(out_path)

In [21]:
def convert_single_pdf_page(pdf_path: str, page: int):
    """
    Convert a single page of a PDF using docling.

    Args:
        pdf_path: Path to the PDF file
        page: Page number to extract (1-based indexing)

    Returns:
        Converted docling Document object containing only the selected page
    """
    if page < 1:
        raise ValueError("Page number must be >= 1 (1-based indexing).")

    pipeline_options = PdfPipelineOptions(
        page_range=(page, page)
    )

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options
            )
        }
    )

    return doc_converter.convert(pdf_path)

In [29]:
for i, row in df_overview_2.iterrows(): 

    if pd.isna(row["Description_page"]):
        continue

    path = os.path.join(positives_pdfs_path, row["Report"])

    description_text, _ = convert_via_single_page_pdf(path, int(row["Description_page"]), description_page_path)
    if description_text is None: 
        continue

    print(row["Report"])
    print(description_text)
    print("-----" * 5 + "\n"*6)

    df_overview_2.loc[i, "Description"] = description_text

Output already exists: projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/company_descriptions/Cryomass Technologies Inc.1__p2.pdf (set overwrite=True to replace)
Output already exists: projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/company_descriptions/Gene Bio Tech Co., Ltd.1__p5.pdf (set overwrite=True to replace)
Output already exists: projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/company_descriptions/Liminal BioSciences Inc.1__p56.pdf (set overwrite=True to replace)
Output already exists: projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/company_descriptions/Gensight Biologics SA1__p1.pdf (set overwrite=True to replace)
Output already exists: projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/company_descriptions/ams-OSR

2025-12-18 10:35:29,880 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-18 10:35:29,882 - INFO - Going to convert document batch...
2025-12-18 10:35:29,883 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-12-18 10:35:29,884 - INFO - Accelerator device: 'cpu'
2025-12-18 10:35:31,344 - INFO - Accelerator device: 'cpu'
2025-12-18 10:35:32,618 - INFO - Accelerator device: 'cpu'
2025-12-18 10:35:32,864 - INFO - Processing document Boustead Singapore Limited1__p3.pdf
2025-12-18 10:35:37,737 - INFO - Finished converting document Boustead Singapore Limited1__p3.pdf in 7.86 sec.
2025-12-18 10:35:37,760 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-18 10:35:37,762 - INFO - Going to convert document batch...
2025-12-18 10:35:37,762 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-12-18 10:35:37,763 - INFO - Accelerator device: 'cpu'


Boustead Singapore Limited1.pdf
## Corporate Profile

Established in 1828, Boustead Singapore Limited (SGX:F9D) is a progressive global InfrastructureRelated Engineering and Technology Group listed on the SGX Mainboard.

As Singapore's oldest continuous business organisation, we focus on the niche engineering and development of key infrastructure to support sustainable shared socio-economic growth. Our strong suite of engineering services under our Energy Engineering Division and Real Estate Division centres on energy infrastructure and smart, eco-sustainable and futureready real estate developments.

In addition, we provide technologydriven transformative solutions to improve the quality of life for all walks of life. Our Geospatial Division provides professional services and exclusively distributes Esri ArcGIS technology - the world's leading geographic information system, smart mapping and location analytics enterprise platform - to major markets in the Asia Pacific. The enterprise 

2025-12-18 10:35:39,220 - INFO - Accelerator device: 'cpu'
2025-12-18 10:35:40,321 - INFO - Accelerator device: 'cpu'
2025-12-18 10:35:40,573 - INFO - Processing document JGC Holdings Corporation1__p18.pdf
/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/nace_git/lib/python3.10/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
2025-12-18 10:36:01,095 - INFO - Finished converting document JGC Holdings Corporation1__p18.pdf in 23.34 sec.
2025-12-18 10:36:01,153 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-18 10:36:01,155 - INFO - Going to convert document batch...
2025-12-18 10:36:01,156 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-12-18 10:36:01,157 - INFO - Accelerator device: 'cpu'


JGC Holdings Corporation1.pdf
Other business

1.4%

## JGC Group at a Glance

Through business focused on Total engineering and Functional materials manufacturing, the JGC Group is aiming to realize our purpose in 'Enhancing planetary health,' and ensuring continued growth of corporate value.

## Breakdown of Sales (Fiscal 2021)

<!-- image -->

|                                     | Segment                                                                                    | Covered Sectors                                                                                                                                                                                                                                                                                                                                                          |
|-------------------------------------|--------------------------------------------------------------------------------------------|------------------------

...

Blackbird PLC1.pdf
- o Net loss after tax £1,917k (12 months to 31 December 2021: net loss after tax £2,135k) due to a worse EBITDA pre LTIP provision and share option costs offset by the LTIP movement and a higher R&amp;D tax credit in 2022 compared to the prior year
- o Net cash outflow, ignoring proceeds from share issues and transfers into short-term deposits, increased to £2,746k (12 months to 31 December 2021: £1,468k) due to £793k costs associated with our Blackbird SaaS platform and £491k increase in trade debtors due to timing of payments from a few customers (these balances were settled in 2023)
- o At 31 December 2022 the Company had cash and short-term deposits of £10,099k (2021: £12,839k) and no debt

## Enquiries:

## Blackbird plc

Tel: +44 (0)20 8879 7245

Ian McDonough, Chief Executive Officer Stephen White, Chief Operating and Financial Officer

## Allenby Capital Limited (Nominated Adviser and Broker)

Tel: +44 (0)20 3328 5656

Nick Naylor / Piers Shimwell (Corporate

2025-12-18 10:38:02,761 - INFO - Accelerator device: 'cpu'
2025-12-18 10:38:03,892 - INFO - Accelerator device: 'cpu'
2025-12-18 10:38:04,146 - INFO - Processing document Expensify, Inc.1__p10.pdf
2025-12-18 10:38:04,900 - INFO - Finished converting document Expensify, Inc.1__p10.pdf in 3.59 sec.
2025-12-18 10:38:04,919 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-18 10:38:04,921 - INFO - Going to convert document batch...
2025-12-18 10:38:04,922 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-12-18 10:38:04,922 - INFO - Accelerator device: 'cpu'


Expensify, Inc.1.pdf
## Part I.

## Item 1. Business

## OVERVIEW

Expensify is a cloud-based expense management software platform that helps the smallest to the largest businesses simplify the way they manage money. Every day, people from all walks of life in organizations around the world use Expensify to scan and reimburse receipts from flights, hotels, coffee shops, office supplies and ride shares. Since our founding in 2008, we have added over 11 million members to our community, and processed and automated over 1.2 billion expense transactions on our platform, freeing people to spend less time managing expenses and more time doing the things they love. For the quarter ended December 31, 2021, an average of 711,000 paid members across 53,000 companies and over 200 countries and territories used Expensify to make money easy.

Small and medium businesses ('SMBs') are the cornerstone of the global economy, making up almost all businesses and the majority of employment in OECD countri

2025-12-18 10:38:06,369 - INFO - Accelerator device: 'cpu'
2025-12-18 10:38:07,494 - INFO - Accelerator device: 'cpu'
2025-12-18 10:38:07,727 - INFO - Processing document Gamelancer Media Corp1__p6.pdf
2025-12-18 10:38:08,418 - INFO - Finished converting document Gamelancer Media Corp1__p6.pdf in 3.50 sec.


Gamelancer Media Corp1.pdf
<!-- image -->

## GENERAL DEVELOPMENT OF THE BUSINESS

Gamelancer is a media and entertainment company that structures and builds creative marketing campaigns for brands and owns and operates a social media network of over 54 channels across TikTok,  Snapchat  and  Instagram  where  it  sells  direct  advertising  for  clients  and  partners ( Gamelancer Media ). Gamelancer develops brand, agency and creator relationships through its strategic partnerships with TikTok North America and Snap Inc., and operates a business with Snapchat where it creates Snapchat Discover shows which generate monthly recurring revenue.

## Three-Year History

## Acquisitions:

## Wondr Gaming Corporation (May 3, 2021):

In  connection  with  the  listing  of  the  Company's  Common  Shares  on  the  CSE,  the  Company completed  a  three-cornered  amalgamation  (the  ' Wondr  Transaction ')  with  Wondr  Gaming Corporation, formerly 1Wondr Gaming Corporation (' Wondr '),  pursua

In [ ]:
df_overview_2 = df_overview_2.rename({"Description_page": "description_page"})

In [31]:
df_overview_2[pd.notna(df_overview_2["Description"])]

,Symbol,Description_page,Unnamed: 0,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,...,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3,NACE_lvl_2,Description
85,KR7086060001,5.0,97,"Gene Bio Tech Co., Ltd.",1,2000.0,KOR,Y2684U105,20060428.0,KOR,...,1.0,086060-KR,1,SHARE,B11R103,A,"Gene Bio Tech Co., Ltd.1.pdf",1.6,1,## WHO WE ARE\n\nBio-Gene is an Australian ...
186,CA53272L2021,56.0,200,Liminal BioSciences Inc.,1,1994.0,USA,53272L202,20120515.0,CAN,...,1.0,LMNL-US,1,SHARE,BQXLM50,C,Liminal BioSciences Inc.1.pdf,21.1,21,## B. Business Overview\n\nWe are a developmen...
189,FR0013183985,1.0,203,Gensight Biologics SA,1,2012.0,FRA,F4374K117,20160713.0,FRA,...,1.0,SIGHT-FR,1,SHARE,BD07M05,C,Gensight Biologics SA1.pdf,21.2,21,<!-- image -->\n\n## GenSight Biologics Report...
192,AT0000A18XM4,12.0,206,ams-OSRAM AG,1,1981.0,CHE,A0400Q115,20080423.0,AUT,...,1.0,AUKUF-US,0,SHARE,BPFJ772,C,ams-OSRAM AG1.pdf,26.1,26,## Our company\n\nams OSRAM is a global leader...
194,KYG4644K1022,9.0,208,Hua Medicine Ltd.,1,2009.0,HKG,G4644K102,20180914.0,CHN,...,1.0,2552-HK,1,SHARE,BF3S3D9,C,Hua Medicine Ltd.1.pdf,21.2,21,## Business overview\n\nWe are a China-based d...
201,MYQ0128OO007,13.0,215,Frontken Corp. Bhd.,1,1996.0,MYS,Y26510100,20060711.0,MYS,...,1.0,0128-MY,1,SHARE,B18TLC4,C,Frontken Corp. Bhd.1.pdf,26.1,26,## CHAPTER 1.0 INTRODUCTION\n\n## 1.1 ABOUT FR...
205,US0326541051,6.0,219,"Analog Devices, Inc.",0,1965.0,USA,032654105,19981201.0,USA,...,1.0,ANL-DE,0,SHARE,5579130,C,"Analog Devices, Inc.1.pdf",26.1,26,"## ITEM 1. BUSINESS\n\n## Company Overview,..."
241,US74275C2052,4.0,255,"Processa Pharmaceuticals, Inc.",1,2011.0,USA,74275C205,20131113.0,USA,...,1.0,PCSA-US,1,SHARE,BL3SW90,C,"Processa Pharmaceuticals, Inc.2.pdf",21.2,21,## PROSPECTUS SUMMARY\n\nThis summary highligh...
243,KYG9005B1041,12.0,257,Transcenta Holding Limited,1,NaN,HKG,G9005B104,20210929.0,CHN,...,1.0,6628-HK,1,SHARE,BMHT0W5,C,Transcenta Holding Limited1.pdf,21.2,21,## OVERVIEW\n\nWe are a clinical stage biophar...
244,US23954D1090,4.0,258,"Day One Biopharmaceuticals, Inc.",1,2018.0,USA,23954D109,20210527.0,USA,...,1.0,DAWN-US,1,SHARE,BLB0YH0,C,"Day One Biopharmaceuticals, Inc.1.pdf",21.2,21,## Item 1. Business.\n\n## Overview\n\nDay One...


In [38]:
df_overview_2_with_description = pd.merge(df_overview_2, df_overview_3[["Symbol", "Description"]], on="Symbol", how="left")

In [39]:
df_overview_2_with_description

,Symbol,Description_page,Unnamed: 0,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,...,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3,NACE_lvl_2,Description_x,Description_y
0,CA05335P1099,NaN,0,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,...,XLY-CA,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf,1.3,1,NaN,NaN
1,JP3947800003,NaN,1,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,...,MMSBF-US,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf",1.4,1,NaN,NaN
2,ID1000167901,NaN,2,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,...,ASHA-ID,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf,3.1,3,NaN,## Riwayat Singkat Perseroan\n\nThe Company at...
3,JP3843250006,NaN,3,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,...,1379-JP,1,SHARE,6432715,A,Hokuto Corporation1.pdf,1.3,1,NaN,NaN
4,VN000000VTQ6,NaN,4,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,...,VTQ-VN,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf,2.3,2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3109,KYG4707P1054,NaN,3278,Icon Culture Global Company Limited,1,2019.0,HKG,G4707P105,20200114.0,CHN,...,8500-HK,1,SHARE,BJYFY12,T,Icon Culture Global Company Limited2.pdf,98.1,98,NaN,NaN
3110,ZAE000015277,NaN,12259,Brimstone Investment Corporation Limited,1,1995.0,ZAF,S13750112,19980708.0,ZAF,...,BRT-ZA,0,SHARE,6119966,A,Brimstone Investment Corporation Limited1.pdf,3.1,3,NaN,NaN
3111,NO0011013765,NaN,31130,Gigante Salmon AS,1,2001.0,NOR,R2724U105,20210705.0,NOR,...,GIGA-NO,1,SHARE,BMC4Z19,A,Gigante Salmon AS1.pdf,3.1,3,NaN,NaN
3112,FR0010776617,NaN,68675,Sapmer SA,1,1989.0,FRA,F7887Q109,20090708.0,FRA,...,ALMER-FR,1,SHARE,B3LS274,A,Sapmer SA2.pdf,3.1,3,NaN,NaN


In [45]:
len(df_overview_2_with_description[pd.notna(df_overview_2_with_description["Description_x"])]), len(df_overview_2_with_description[pd.notna(df_overview_2_with_description["Description_y"])])

KeyboardInterrupt: 

In [44]:
len(df_overview_2_with_description[pd.notna(df_overview_2_with_description["Description_x"]) & pd.notna(df_overview_2_with_description["Description_y"])])

23

In [48]:
df_overview_2_with_description

,Symbol,Description_page,Unnamed: 0,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,...,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3,NACE_lvl_2,Description_x,Description_y,Description
0,CA05335P1099,NaN,0,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,...,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf,1.3,1,NaN,NaN,NaN
1,JP3947800003,NaN,1,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,...,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf",1.4,1,NaN,NaN,NaN
2,ID1000167901,NaN,2,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,...,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf,3.1,3,NaN,## Riwayat Singkat Perseroan\n\nThe Company at...,NaN
3,JP3843250006,NaN,3,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,...,1,SHARE,6432715,A,Hokuto Corporation1.pdf,1.3,1,NaN,NaN,NaN
4,VN000000VTQ6,NaN,4,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,...,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf,2.3,2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3109,KYG4707P1054,NaN,3278,Icon Culture Global Company Limited,1,2019.0,HKG,G4707P105,20200114.0,CHN,...,1,SHARE,BJYFY12,T,Icon Culture Global Company Limited2.pdf,98.1,98,NaN,NaN,NaN
3110,ZAE000015277,NaN,12259,Brimstone Investment Corporation Limited,1,1995.0,ZAF,S13750112,19980708.0,ZAF,...,0,SHARE,6119966,A,Brimstone Investment Corporation Limited1.pdf,3.1,3,NaN,NaN,NaN
3111,NO0011013765,NaN,31130,Gigante Salmon AS,1,2001.0,NOR,R2724U105,20210705.0,NOR,...,1,SHARE,BMC4Z19,A,Gigante Salmon AS1.pdf,3.1,3,NaN,NaN,NaN
3112,FR0010776617,NaN,68675,Sapmer SA,1,1989.0,FRA,F7887Q109,20090708.0,FRA,...,1,SHARE,B3LS274,A,Sapmer SA2.pdf,3.1,3,NaN,NaN,NaN


10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
00{"name":"Exception","message":"not_supported","stack":["Traceback \u001b(most recent call last)\u001b:\n","\u001b  File \u001b<string>:174\u001b in \u001b__DW_DEBUG_WRAPPER__\u001b\n","\u001b  File \u001b<string>:165\u001b in \u001b__DW_GET_EXPRESSION_VARI

In [ ]:
df_overview_2_with_description["Description"] = df_overview_2_with_description["Description_x"]
pd.notna(df_overview_2_with_description["Description"]).sum()

60

10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\

In [58]:
df_overview_2_with_description.loc[pd.isna(df_overview_2_with_description["Description"]),"Description"] = df_overview_2_with_description[pd.isna(df_overview_2_with_description["Description"])]["Description_y"]

10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}


In [59]:
pd.notna(df_overview_2_with_description["Description"]).sum()

95

10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}


In [ ]:
df_overview_2

,Symbol,Description_page,Unnamed: 0,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,...,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3,NACE_lvl_2,Description
0,CA05335P1099,NaN,0,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,...,1.0,XLY-CA,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf,1.3,1,NaN
1,JP3947800003,NaN,1,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,...,1.0,MMSBF-US,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf",1.4,1,NaN
2,ID1000167901,NaN,2,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,...,1.0,ASHA-ID,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf,3.1,3,NaN
3,JP3843250006,NaN,3,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,...,1.0,1379-JP,1,SHARE,6432715,A,Hokuto Corporation1.pdf,1.3,1,NaN
4,VN000000VTQ6,NaN,4,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,...,1.0,VTQ-VN,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf,2.3,2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3109,KYG4707P1054,NaN,3278,Icon Culture Global Company Limited,1,2019.0,HKG,G4707P105,20200114.0,CHN,...,1.0,8500-HK,1,SHARE,BJYFY12,T,Icon Culture Global Company Limited2.pdf,98.1,98,NaN
3110,ZAE000015277,NaN,12259,Brimstone Investment Corporation Limited,1,1995.0,ZAF,S13750112,19980708.0,ZAF,...,1.0,BRT-ZA,0,SHARE,6119966,A,Brimstone Investment Corporation Limited1.pdf,3.1,3,NaN
3111,NO0011013765,NaN,31130,Gigante Salmon AS,1,2001.0,NOR,R2724U105,20210705.0,NOR,...,1.0,GIGA-NO,1,SHARE,BMC4Z19,A,Gigante Salmon AS1.pdf,3.1,3,NaN
3112,FR0010776617,NaN,68675,Sapmer SA,1,1989.0,FRA,F7887Q109,20090708.0,FRA,...,1.0,ALMER-FR,1,SHARE,B3LS274,A,Sapmer SA2.pdf,3.1,3,NaN


10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
00{"name":"Exception","message":"not_supported","stack":["Traceback \u001b(most recent call last)\u001b:\n","\u001b  File \u001b<string>:174\u001b in \u001b__DW_DEBUG_WRAPPER__\u001b\n","\u001b  File \u001b<string>:165\u001b in \u001b__DW_GET_EXPRESSION_VARIABLE__\u001b\n","\u001bException\u001b\u001b:\u001b not_supported\n"]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\"

In [61]:
df_overview_2_with_description.to_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_overview_with_description.csv", sep=",")

10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\